# Stacking classification and Regression

## -> Stacking Classification

In [15]:
from sklearn.ensemble import StackingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [17]:
x,y=make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=1)

In [18]:
x

array([[-2.04582165, -0.13791624, -0.08071423, ...,  2.48194524,
         0.74236675,  0.23154789],
       [-0.98726024,  1.30120189,  2.37734888, ...,  0.55445754,
        -0.21892143, -0.37608578],
       [ 0.57335921,  0.09375582,  0.4662521 , ..., -0.6088508 ,
         0.79903499, -0.17121177],
       ...,
       [-0.70737159,  1.07650943,  0.58510456, ..., -1.51337602,
         0.90239871, -0.69230951],
       [-0.20706849,  1.17319848, -1.94478665, ..., -0.32820676,
         1.5711921 ,  1.14877729],
       [-2.16769231, -2.54871672,  2.89359255, ...,  0.71535366,
         0.34329241,  1.07350284]])

In [19]:
y

array([0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0,
       0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0,
       0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0,
       0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [20]:
x_train,x_test,y_train,y_test=train_test_split(x, y, test_size=0.3, random_state=1)

In [22]:
base_model=[
    ('xgb',XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=1)),
    ('catboost',CatBoostClassifier(iterations=100, learning_rate=0.1, depth=5, verbose=0, random_state=1)),
    ('decision_tree',DecisionTreeClassifier(max_depth=5, random_state=1))
]

In [25]:
# Meta model (final estimator)
meta_model=LogisticRegression()

In [26]:
# Stacking Classifier
stacking_classifier=StackingClassifier(estimators=base_model, final_estimator=meta_model, cv=5)
stacking_classifier

StackingClassifier(cv=5,
                   estimators=[('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_...
                                              max_depth=None, max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=100, n_jobs=None,
                                              num_parallel_tree=None, ...)),
                               ('catboost',
                                CatBoostClassifier(depth=5, iterations=100, learning_rate=0.1, random_state=1, verbose=0)),
                               ('decision_tree',
                                DecisionTreeClassifier(max_depth=5,
                                                       random_state=1))],
                   final_estimator=LogisticRegression())

In [27]:
stacking_classifier.fit(x_train, y_train)

# Predict
y_pred=stacking_classifier.predict(x_test)

# Evaliuate the model
print("Stacking classifier performance:")
print(f"Accuracy:{accuracy_score(y_test, y_pred)}")
print("Classification report")
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Stacking classifier performance:
Accuracy:0.86
Classification report
              precision    recall  f1-score   support

           0       0.84      0.86      0.85       139
           1       0.88      0.86      0.87       161

    accuracy                           0.86       300
   macro avg       0.86      0.86      0.86       300
weighted avg       0.86      0.86      0.86       300

Confusion matrix:
[[120  19]
 [ 23 138]]


## -> Stacking Regressor

In [31]:
from sklearn.ensemble import StackingRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

In [32]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [33]:
x, y=make_regression(n_samples=1000, n_features=10, noise=10, random_state=1)

In [34]:
x

array([[-1.24634541, -2.3575232 ,  0.60972409, ..., -1.25935848,
        -0.11048061,  0.46983129],
       [-0.68085157, -1.06787658,  0.57296273, ..., -0.01781755,
         0.45794708, -0.6001388 ],
       [-0.14894123,  1.16533544, -0.63259014, ...,  1.89716069,
        -0.20984695, -1.38139115],
       ...,
       [ 1.32960903, -1.08278525,  0.44347873, ..., -0.48363166,
         0.01880501,  0.56264832],
       [ 1.03703898,  0.67261975,  1.00568668, ...,  0.61472628,
         0.35356722, -0.34898419],
       [ 0.438562  ,  0.92781985,  0.72667997, ..., -1.09330391,
        -0.37195994,  0.22445073]])

In [35]:
y

array([-133.24505709,  -32.3653315 , -247.5595513 ,  -47.1235069 ,
         34.48691149,   15.83930904, -168.27724002,   89.29224394,
        -28.17709411, -201.18957616, -105.97320089, -188.71955284,
       -167.20011997,  187.48804495,   52.28930514, -352.30400411,
         73.44665853,  161.50918083,  157.41717265,  508.53558642,
        134.2897571 ,   42.91791473, -191.01661258,  -42.48258988,
       -180.90912039,   45.06136385,  -61.31732606,  -50.87025004,
       -114.23068412,  223.83580519,  260.89185092, -183.38111923,
        304.5432027 ,  -21.64449415,  426.22157479,   96.30622856,
       -129.84724217,  335.8807279 , -157.1840908 ,  356.47185108,
        197.53520647, -175.8551562 ,  120.17294555, -308.45194864,
       -114.36899919,   45.85563769,  158.23767227,   78.29299732,
        -58.15212938,  -32.20904183,  345.20517107,  116.90804324,
       -205.45985542,  -65.08617302, -133.63600593,    9.19544299,
        -55.3661145 ,  -79.24837934, -143.94624434,  -46.15031

In [36]:
x_train,x_test,y_train,y_test=train_test_split(x, y, test_size=0.3, random_state=1)

In [37]:
x_train.shape, x_test.shape

((700, 10), (300, 10))

In [40]:
base_model=[
    ('xgb',XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=1)),
    ('catboost',CatBoostRegressor(iterations=100, learning_rate=0.1, depth=5, verbose=0, random_state=1)),
    ('decision_tree',DecisionTreeRegressor(max_depth=5, random_state=1))
]

# Meta model (final estimator)
meta_model=Ridge()

In [41]:
# Stacking Classifier
stacking_regressor=StackingRegressor(estimators=base_model, final_estimator=meta_model, cv=5)
stacking_regressor

StackingRegressor(cv=5,
                  estimators=[('xgb',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_co...
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=100, n_jobs=None,
                                            num_parallel_tree=None, ...)),
                              ('catboost',
                               CatBoostRegressor(depth=5, iterations=100, learning_rate=0.1, loss_function='RMSE', random_state=1, verbose=0)),
                              ('decision_tree',
                               DecisionTreeRegressor(max_depth=5,
                                                     random_state=1))],
                  final_estimator=Ridge())

In [42]:
stacking_regressor.fit(x_train, y_train)

# Predict
y_pred=stacking_regressor.predict(x_test)

# Evaliuate the model
print('Stacking Regressor performance:')
print(f'R2 score:{r2_score(y_test, y_pred)}')
print(f'Mean Absolute Error:{mean_absolute_error(y_test, y_pred)}')
print(f'Mean Squared error:{mean_squared_error(y_test, y_pred)}')

Stacking Regressor performance:
R2 score:0.966824191889789
Mean Absolute Error:24.874969155918723
Mean Squared error:1097.6389833769201
